In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import subprocess

In [2]:
sys.path.append("../../../training_data")

In [3]:
from utils.utils import Cif

In [4]:
with open("../../../training_data/7.Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pd.read_pickle(f)

len(extras_featuresd), extras_featuresd

(21,
 {'22mj':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       22mj               1             A           10            A   
  1       22mj               1             A           11            A   
  2       22mj               1             A           12            A   
  3       22mj               1             A           13            A   
  4       22mj               1             A           14            A   
  ..       ...             ...           ...          ...          ...   
  281     22mj               1             A          315            A   
  282     22mj               1             A          316            A   
  283     22mj               1             A          317            A   
  284     22mj               1             A          318            A   
  285     22mj               1             A          319            A   
  
                      

# Make predictions

Edits throughout to fix:
- Hardcoded paths
- Pass locations of ProtT5 and the nr database

# Process

In [5]:
results = {}

for pdb, feats in extras_featuresd.items():
    # if pdb == "8aq6": continue
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    resf = f"{pdb}/{pdb}_allosteric_residues.txt"
    if os.path.isfile(resf):
        with open(resf) as f:
            txt = f.read()
        chain = txt.split("Chain", 1)[1].strip().split()[0]
        resids = [x for x in txt.split("resid", 1)[1].replace("(", "").replace(")", "").replace(",", " ").split() if x.isdigit()]   

        results[pdb.lower()] = {"pocket": {"residues": (
            pd.DataFrame({"auth_asym_id": [chain]*len(resids), "auth_seq_id": resids}, dtype=str)
            .merge(Cif(pdb, f"../structures/{pdb}.cif").residues)
            [["auth_asym_id", "auth_seq_id"]]
        )}}

len(results), results

(21,
 {'22mj': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A          87
    1            A          93
    2            A          96
    3            A         118
    4            A         124
    5            A         268
    6            A         304}},
  '6s3a': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         132
    1            A         135
    2            A         136
    3            A         176
    4            A         177
    5            A         190
    6            A         218
    7            A         221}},
  '6vvq': {'pocket': {'residues':    auth_asym_id auth_seq_id
    0             C         426
    1             C         428
    2             C         449
    3             C         452
    4             C         490
    5             C         528
    6             C         530
    7             C         532
    8             C         546
    9             C         548
    10            C 

In [6]:
pd.to_pickle(results, "allofusion_results.pkl")